In [1]:
import pandas as pd
import numpy as np

nav = pd.read_csv(
    r"C:\Users\Dell\OneDrive\Desktop\Fintech_Data_Analytics\Data\raw\02_nav_history.csv"
)

nav["date"] = pd.to_datetime(nav["date"])

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

In [4]:
nav.head(10)

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210
5755,100016,2022-01-10,510.7136,-0.008639
5756,100016,2022-01-11,513.5542,0.005562
5757,100016,2022-01-12,512.3195,-0.002404
5758,100016,2022-01-13,510.2445,-0.004050
5759,100016,2022-01-14,514.3636,0.008073


In [5]:
volatility = (
    nav.groupby("amfi_code")["daily_return"]
       .std()
       * np.sqrt(252)
)

volatility.head()

amfi_code
100016    0.145481
100025    0.039052
100033    0.189367
101206    0.145682
101207    0.257973
Name: daily_return, dtype: float64

In [6]:
nav["daily_return"].isna().sum()

np.int64(40)

In [7]:
annual_return = (
    nav.groupby("amfi_code")["daily_return"]
       .mean()
       * 252
)

annual_return.head()

amfi_code
100016    0.035683
100025    0.042854
100033    0.272111
101206    0.214647
101207    0.106962
Name: daily_return, dtype: float64

In [8]:
risk_free_rate = 0.06

sharpe_ratio = (
    annual_return - risk_free_rate
) / volatility

sharpe_ratio.head()

amfi_code
100016   -0.167148
100025   -0.439062
100033    1.120102
101206    1.061535
101207    0.182043
Name: daily_return, dtype: float64

In [9]:
metrics = pd.DataFrame({
    "annual_return": annual_return,
    "volatility": volatility,
    "sharpe_ratio": sharpe_ratio
})

metrics.head()

,annual_return,volatility,sharpe_ratio
amfi_code,,,
100016,0.035683,0.145481,-0.167148
100025,0.042854,0.039052,-0.439062
100033,0.272111,0.189367,1.120102
101206,0.214647,0.145682,1.061535
101207,0.106962,0.257973,0.182043


In [10]:
metrics.sort_values(
    "sharpe_ratio",
    ascending=False
).head(10)

,annual_return,volatility,sharpe_ratio
amfi_code,,,
120507,0.067448,0.004939,1.508048
148567,0.270566,0.141937,1.483518
120843,0.272602,0.158870,1.338216
148569,0.283262,0.176740,1.263220
119551,0.231033,0.137414,1.244653
120505,0.292653,0.192909,1.206020
149323,0.265908,0.177462,1.160297
100033,0.272111,0.189367,1.120102
118632,0.218037,0.141484,1.116999


Funds with higher Sharpe ratios provide better risk-adjusted returns

In [11]:
def max_drawdown(series):

    cumulative = (1 + series).cumprod()

    rolling_max = cumulative.cummax()

    drawdown = (
        cumulative - rolling_max
    ) / rolling_max

    return drawdown.min()

In [12]:
max_dd = (
    nav.groupby("amfi_code")["daily_return"]
       .apply(max_drawdown)
)

max_dd.head()

amfi_code
100016   -0.247344
100025   -0.043083
100033   -0.162172
101206   -0.112916
101207   -0.354469
Name: daily_return, dtype: float64

In [13]:
var_95 = (
    nav.groupby("amfi_code")["daily_return"]
       .quantile(0.05)
)

var_95.head()

amfi_code
100016   -0.014364
100025   -0.003793
100033   -0.019034
101206   -0.013282
101207   -0.026021
Name: daily_return, dtype: float64

VaR estimates the worst expected one-day loss under normal market conditions at a 95% confidence level.


In [14]:
metrics["max_drawdown"] = max_dd
metrics["var_95"] = var_95

metrics.head()

,annual_return,volatility,sharpe_ratio,max_drawdown,var_95
amfi_code,,,,,
100016,0.035683,0.145481,-0.167148,-0.247344,-0.014364
100025,0.042854,0.039052,-0.439062,-0.043083,-0.003793
100033,0.272111,0.189367,1.120102,-0.162172,-0.019034
101206,0.214647,0.145682,1.061535,-0.112916,-0.013282
101207,0.106962,0.257973,0.182043,-0.354469,-0.026021
